# 🧪 Benchmark d'Évaluation — Pipeline RAG Neurosymbolique (Briques 1→5)

**Objectif** : Évaluer la précision de notre pipeline IA sur les réponses réelles de **5 cardiologues** × **15 cas ECG**.

**Score** = moyenne des % des **diagnostics validants** matchés (les descripteurs sont trackés séparément).

**Pipeline testé** :
1. **Brique 2** — Extraction NER (GPT-4o Structured Outputs)
2. **Brique 3** — Recherche Hybride (Dense + BM25 + RRF)
3. **Brique 4** — Juge Neurosymbolique (Coupe-Circuit + GPT-4o-mini QCM)
4. **Scoring** — Validants : EXACT (100%) / CHILD (90%) / PARENT (40%) / MISSING (0%)

| Cellule | Section | Description |
|---------|---------|-------------|
| 1 | **Setup** | Imports + Initialisation moteur RAG |
| 2 | **Données** | Chargement CSV + Golden Set (validants / descripteurs) |
| 3 | **Pipeline** | Boucle de traitement (score = validants uniquement) |
| 4 | **Audit** | Heatmap + Audit détaillé (code couleur match_type) |

In [7]:
# ============================================================
# CELLULE 1 — Imports & Initialisation du Moteur RAG
# ============================================================
import sys, os, json, time, warnings, importlib
import pandas as pd
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
from tqdm.auto import tqdm

warnings.filterwarnings('ignore', category=FutureWarning)

# ─── Chemins racines ──────────────────────────────────────────────────────────
PROJECT_ROOT = Path(r"C:\Users\Administrateur\bmad\ECG lecture")
EVAL_ROOT    = Path(r"C:\Users\Administrateur\bmad\ECG evaluation")
RAG_ROOT     = Path(r"C:\Users\Administrateur\bmad\RAG ontologique")

# Ajouter au PYTHONPATH pour les imports
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(RAG_ROOT))

# Charger la clé API
load_dotenv(PROJECT_ROOT / ".env")

# ─── Imports RAG Neurosymbolique (Briques 2, 3, 4) ───────────────────────────
# Force reload pour utiliser les versions modifiées sur disque
import ner_extractor; importlib.reload(ner_extractor)
import ontology_index; importlib.reload(ontology_index)
import hybrid_search; importlib.reload(hybrid_search)
import neurosymbolic_judge; importlib.reload(neurosymbolic_judge)

from ner_extractor import extract_clinical_terms, NERExtraction
from hybrid_search import HybridSearchEngine
from neurosymbolic_judge import resolve_term_to_ontology

# ─── Import du scoring de production (find_owl_concept + implications) ────────
import frontend.pages.correction_llm; importlib.reload(frontend.pages.correction_llm)
from frontend.pages.correction_llm import find_owl_concept, apply_implication_rules

# ─── Initialiser le moteur de recherche hybride en RAM (une seule fois) ───────
os.chdir(str(RAG_ROOT))  # Pour que HybridSearchEngine trouve rag_index/
moteur = HybridSearchEngine()

# ─── Charger l'ontologie (pour les poids / catégories) ───────────────────────
with open(PROJECT_ROOT / "data" / "ontology_from_owl.json", 'r', encoding='utf-8') as f:
    ONTOLOGY = json.load(f)

CONCEPT_MAPPINGS = ONTOLOGY.get('concept_mappings', {})
IMPLICATION_RULES = ONTOLOGY.get('implication_rules', {})

# ─── Vérifications ───────────────────────────────────────────────────────────
print(f"[OK] OPENAI_API_KEY : {'✅' if os.getenv('OPENAI_API_KEY') else '❌'}")
print(f"[OK] HybridSearchEngine : {len(moteur.documents)} documents indexés")
print(f"[OK] Ontologie : {len(CONCEPT_MAPPINGS)} concepts chargés")
print(f"[OK] Implications : {len(IMPLICATION_RULES)} règles")

# Vérification clé : TV est-il indexé pour TACHYCARDIE_VENTRICULAIRE ?
tv_forms = moteur.get_all_normalized_forms("TACHYCARDIE_VENTRICULAIRE")
print(f"[OK] TACHYCARDIE_VENTRICULAIRE formes: {tv_forms}")
print(f"[OK] 'tv' dans formes: {'tv' in tv_forms}")

print(f"\n🚀 Moteur RAG prêt — Briques 2+3+4 opérationnelles (modules rechargés).")

[OK] OPENAI_API_KEY : ✅
[OK] HybridSearchEngine : 522 documents indexés
[OK] Ontologie : 288 concepts chargés
[OK] Implications : 0 règles
[OK] TACHYCARDIE_VENTRICULAIRE formes: {'tv', 'vt', 'tachycardie ventriculaire'}
[OK] 'tv' dans formes: True

🚀 Moteur RAG prêt — Briques 2+3+4 opérationnelles (modules rechargés).


## 📂 Cellule 2 — Chargement des Données Réelles

- **CSV** : Réponses de 5 cardiologues sur 15 cas ECG (`ECG_Collector_Data.csv`)
- **Golden Set** : 15 dossiers avec `metadata.json` (annotations expert, diagnostic principal, poids)

Le DataFrame final (`df_flat`) contiendra une ligne par **(participant × cas)** avec le texte brut et le golden set attendu.

In [8]:
# ============================================================
# CELLULE 2 — Chargement CSV + Golden Set
# ============================================================

# ─── A) Réponses des collègues ───────────────────────────────────────────────
df_responses = pd.read_csv(EVAL_ROOT / "ECG_Collector_Data.csv")
PARTICIPANTS = df_responses['code'].tolist()
print(f"[CSV] {len(PARTICIPANTS)} participants : {PARTICIPANTS}")
print(f"[CSV] Colonnes : {list(df_responses.columns)}")

# ─── B) Golden Set (15 cas) ──────────────────────────────────────────────────
golden_cases = {}
for case_dir in sorted((EVAL_ROOT / "goldenset").iterdir()):
    meta_file = case_dir / "metadata.json"
    if case_dir.is_dir() and meta_file.exists():
        with open(meta_file, 'r', encoding='utf-8') as f:
            meta = json.load(f)
        case_num = int(meta['name'])
        golden_cases[case_num] = {
            'case_id': meta['case_id'],
            'diagnostic_principal': meta.get('diagnostic_principal', ''),
            'annotations': meta.get('annotations', []),
            'expected_concepts': meta.get('expected_concepts', []),
        }

ALL_CASES = sorted(golden_cases.keys())
print(f"\n[GOLD] {len(golden_cases)} cas chargés :")
for num, case in sorted(golden_cases.items()):
    validants = [a['concept'] for a in case['annotations']
                 if 'validant' in a.get('annotation_role', '').lower()]
    descripteurs = [a['concept'] for a in case['annotations']
                    if 'validant' not in a.get('annotation_role', '').lower()]
    print(f"   Cas {num:2d} | {case['diagnostic_principal']:<45s} | "
          f"{len(validants)} validants, {len(descripteurs)} descripteurs")

# ─── C) Aplatir en DataFrame (1 ligne = 1 participant × 1 cas) ──────────────
rows = []
for participant in PARTICIPANTS:
    for case_num in ALL_CASES:
        golden = golden_cases[case_num]
        col_name = f'cas_{case_num:02d}'
        raw_text = df_responses.loc[df_responses['code'] == participant, col_name].values[0]

        # Extraire les concepts attendus (validants + descripteurs) avec leur rôle
        expected_ids = []
        expected_names = []
        expected_roles = []  # "validant" ou "descripteur"
        for ann in golden['annotations']:
            owl = find_owl_concept(ann['concept'])
            if owl:
                expected_ids.append(owl['ontology_id'])
                expected_names.append(ann['concept'])
                role = "validant" if "validant" in ann.get('annotation_role', '').lower() else "descripteur"
                expected_roles.append(role)

        rows.append({
            'participant': participant,
            'cas': case_num,
            'diagnostic_principal': golden['diagnostic_principal'],
            'texte_etudiant': str(raw_text).strip() if pd.notna(raw_text) else '',
            'golden_ids': expected_ids,
            'golden_names': expected_names,
            'golden_roles': expected_roles,
        })

df_flat = pd.DataFrame(rows)
df_flat['is_empty'] = df_flat['texte_etudiant'].isin(['', 'nan'])

print(f"\n[DATA] DataFrame : {len(df_flat)} lignes ({len(PARTICIPANTS)} participants × {len(ALL_CASES)} cas)")
print(f"   Textes vides : {df_flat['is_empty'].sum()}")
df_flat.head(10)

[CSV] 5 participants : ['ECG-7512', 'ECG-IDXQ', 'ECG-3RMP', 'ECG-DFLC', 'ECG-1I3Q']
[CSV] Colonnes : ['code', 'cas_01', 'cas_02', 'cas_03', 'cas_04', 'cas_05', 'cas_06', 'cas_07', 'cas_08', 'cas_09', 'cas_10', 'cas_11', 'cas_12', 'cas_13', 'cas_14', 'cas_15']

[GOLD] 15 cas chargés :
   Cas  1 | ECG normal                                    | 1 validants, 1 descripteurs
   Cas  2 | BAV complet                                   | 1 validants, 1 descripteurs
   Cas  3 | Fibrillation atriale                          | 1 validants, 1 descripteurs
   Cas  4 | Microvoltage                                  | 1 validants, 3 descripteurs
   Cas  5 | Hyperkaliémie                                 | 2 validants, 0 descripteurs
   Cas  6 | Stimulation atriale                           | 2 validants, 1 descripteurs
   Cas  7 | Bloc de branche droit complet                 | 3 validants, 0 descripteurs
   Cas  8 | Flutter droit typique                         | 1 validants, 0 descripteurs
   Cas  9 |

,participant,cas,diagnostic_principal,texte_etudiant,golden_ids,golden_names,golden_roles,is_empty
0,ECG-7512,1,ECG normal,Sinusal qrs fins \nP bifide,"[BLOC_INTERATRIAL, ECG_NORMAL]","[Bloc interatrial, ECG normal]","[descripteur, validant]",False
1,ECG-7512,2,BAV complet,bav 1 hbag bbd,"[BAV_COMPLET, ECHAPPEMENT_VENTRICULAIRE]","[BAV complet, Echappement ventriculaire]","[validant, descripteur]",False
2,ECG-7512,3,Fibrillation atriale,fibrillation atriale,"[FIBRILLATION_ATRIALE, REPOLARISATION_PRÉCOCE]","[Fibrillation atriale, Repolarisation précoce]","[validant, descripteur]",False
3,ECG-7512,4,Microvoltage,microvoltage,"[AMYLOSE, BAV_DE_TYPE_1, PERTE_DES_ONDE_Q_SEPT...","[Amylose, BAV de type 1, Perte des onde Q sept...","[descripteur, descripteur, descripteur, validant]",False
4,ECG-7512,5,Hyperkaliémie,hyperkaliemie qrs fins onde amble bav complet,"[HYPERKALIÉMIE, BAV_DE_HAUT_GRADE]","[Hyperkaliémie, BAV de haut grade]","[validant, validant]",False
5,ECG-7512,6,Stimulation atriale,stimulation atriale,"[STIMULATION_ATRIALE, BLOC_FASCICULAIRE_ANTÉRI...","[Stimulation atriale, Bloc fasciculaire antéri...","[validant, validant, descripteur]",False
6,ECG-7512,7,Bloc de branche droit complet,bbd et hbag bav 1,"[BLOC_DE_BRANCHE_DROIT_COMPLET, BAV_DE_TYPE_1,...","[Bloc de branche droit complet, BAV de type 1,...","[validant, validant, validant]",False
7,ECG-7512,8,Flutter droit typique,flutter commun qrs normaux,[FLUTTER_DROIT_TYPIQUE],[Flutter droit typique],[validant],False
8,ECG-7512,9,BAV 2 Mobitz 2,bav 2 mobitz 1,"[BAV_2_MOBITZ_2, BLOC_DE_BRANCHE_DROIT, BLOC_F...","[BAV 2 Mobitz 2, Bloc de branche droit, Bloc f...","[validant, descripteur, descripteur]",False
9,ECG-7512,10,Bloc de branche gauche complet,bloc de branche gauche,"[BLOC_DE_BRANCHE_GAUCHE_COMPLET, RYTHME_SINUSA...","[Bloc de branche gauche complet, Rythme sinusa...","[validant, descripteur, descripteur]",False


## ⚙️ Cellule 3 — Boucle de Traitement (Le Cœur du Benchmark)

Pour chaque **(participant × cas)** :
1. **Brique 2** : `extract_clinical_terms(texte)` → entités NER avec statut
2. **Brique 3** : `moteur.search_top_k(terme_brut)` → Top-K candidats
3. **Brique 4** : `resolve_term_to_ontology(terme, contexte, candidats)` → ID ontologique
4. **Scoring** : Score = **moyenne des % des validants matchés** (descripteurs hors note)

| Match | Score | Signification |
|-------|-------|---------------|
| ✅ EXACT | 100% | ID trouvé = ID attendu |
| 🟠 CHILD | 90% | Concept plus spécifique (enfant) |
| 🔴 PARENT | 40% | Concept plus général (parent) |
| 🔵 IMPL | 100% | Auto-validé par implication |
| ❌ MISSING | 0% | Non trouvé |

⏱️ **Temps estimé** : ~5-10 min

In [9]:
# ============================================================
# CELLULE 3 — Pipeline complet : Extraction → RAG → Scoring
# ============================================================
import importlib
import scoring
importlib.reload(scoring)
from scoring import score_student_response, find_owl_concept, SCORE_CHILD_MATCH, SCORE_PARENT_MATCH

def run_pipeline(texte: str, golden_names: list, golden_ids: list,
                 golden_roles: list, diagnostic_principal: str):
    """
    Pipeline RAG Neurosymbolique complet pour 1 texte étudiant.
    Score = moyenne des % des diagnostics VALIDANTS uniquement.
    """
    result = {
        'nb_entites': 0,
        'entites_extraites': [],
        'ids_trouves': [],
        'noms_trouves': [],
        'statuts': [],
        'methodes': [],
        'matched_expected': [],
        'missing_expected': [],
        'auto_validated': [],
        'score_final_pct': 0.0,
        'latence_s': 0.0,
        'erreur': None,
        'match_types': {},
        'partial_matches': [],
        'validant_found': 0,
        'validant_total': 0,
        'descripteur_found': 0,
        'descripteur_total': 0,
    }
    if not texte or texte in ('', 'nan'):
        result['erreur'] = 'texte_vide'
        return result
    t0 = time.time()
    try:
        # ─── Brique 2 : Extraction NER ───────────────────────────────────
        extraction = extract_clinical_terms(texte)
        result['nb_entites'] = len(extraction.entites)

        # ─── Briques 3+4 : Recherche + Juge ─────────────────────────────
        student_matched_ids = {}
        for entite in extraction.entites:
            result['entites_extraites'].append(entite.terme_brut)
            candidats = moteur.search_top_k(entite.terme_brut)
            resolution = resolve_term_to_ontology(
                entite.terme_brut, entite.contexte_phrase, candidats
            )
            matched_id = resolution["ontology_id"]
            result['methodes'].append(resolution["method"])
            if matched_id != "NONE":
                student_matched_ids[matched_id] = entite.statut
                result['ids_trouves'].append(matched_id)
                result['noms_trouves'].append(resolution.get("concept_name", matched_id))
                result['statuts'].append(entite.statut)

        # ─── Brique 5 : Scoring (validants uniquement) ──────────────────
        scoring_result = score_student_response(
            found_ids=list(student_matched_ids.keys()),
            found_statuts=student_matched_ids,
            golden_names=golden_names,
            golden_ids=golden_ids,
            golden_roles=golden_roles,
        )

        result['matched_expected'] = scoring_result['matched_expected']
        result['missing_expected'] = scoring_result['missing_expected']
        result['auto_validated'] = scoring_result['auto_validated']
        result['score_final_pct'] = scoring_result['score_final_pct']
        result['match_types'] = scoring_result.get('match_types', {})
        result['partial_matches'] = scoring_result.get('partial_matches', [])
        result['validant_found'] = scoring_result['validant_found']
        result['validant_total'] = scoring_result['validant_total']
        result['descripteur_found'] = scoring_result['descripteur_found']
        result['descripteur_total'] = scoring_result['descripteur_total']

    except Exception as e:
        result['erreur'] = str(e)[:120]
    result['latence_s'] = round(time.time() - t0, 2)
    return result

# ─── Boucle principale ───────────────────────────────────────────────────────
print(f"🚀 Benchmark RAG Neurosymbolique : {len(df_flat)} évaluations")
print(f"   {len(PARTICIPANTS)} participants × {len(ALL_CASES)} cas")
print(f"{'='*90}")

all_results = []
all_match_types = []
t_start = time.time()

for idx, row in tqdm(df_flat.iterrows(), total=len(df_flat), desc="Pipeline RAG"):
    res = run_pipeline(
        texte=row['texte_etudiant'],
        golden_names=row['golden_names'],
        golden_ids=row['golden_ids'],
        golden_roles=row['golden_roles'],
        diagnostic_principal=row['diagnostic_principal'],
    )
    all_results.append(res)
    for mt in res.get('match_types', {}).values():
        all_match_types.append(mt)

elapsed = time.time() - t_start

# ─── Intégrer les résultats dans le DataFrame ────────────────────────────────
df_flat['nb_entites'] = [r['nb_entites'] for r in all_results]
df_flat['concepts_ia'] = [' | '.join(r['noms_trouves']) for r in all_results]
df_flat['entites_brutes'] = [' | '.join(r['entites_extraites']) for r in all_results]
df_flat['statuts'] = [' | '.join(r['statuts']) for r in all_results]
df_flat['methodes'] = [' | '.join(r['methodes']) for r in all_results]
df_flat['matched'] = [' | '.join(r['matched_expected']) for r in all_results]
df_flat['missing'] = [' | '.join(r['missing_expected']) for r in all_results]
df_flat['auto_validated'] = [' | '.join(r['auto_validated']) for r in all_results]
df_flat['score_final'] = [r['score_final_pct'] for r in all_results]
df_flat['latence_s'] = [r['latence_s'] for r in all_results]
df_flat['erreur'] = [r['erreur'] for r in all_results]
df_flat['validant_found'] = [r['validant_found'] for r in all_results]
df_flat['validant_total'] = [r['validant_total'] for r in all_results]
df_flat['descripteur_found'] = [r['descripteur_found'] for r in all_results]
df_flat['descripteur_total'] = [r['descripteur_total'] for r in all_results]

# ─── Résumé ──────────────────────────────────────────────────────────────────
df_valid = df_flat[df_flat['erreur'].isna()].copy()

print(f"\n{'='*90}")
print(f"✅ Benchmark terminé en {elapsed:.0f}s ({elapsed/60:.1f} min)")
print(f"   Évaluations réussies : {len(df_valid)} / {len(df_flat)}")
print(f"   Score moyen (validants)  : {df_valid['score_final'].mean():.1f}%")
print(f"   Score médian             : {df_valid['score_final'].median():.1f}%")
print(f"   Écart-type               : {df_valid['score_final'].std():.1f}%")
print(f"   Latence moy/cas          : {df_valid['latence_s'].mean():.1f}s")

# Stats méthodes
all_methodes = [m for r in all_results for m in r['methodes']]
n_cc = all_methodes.count('coupe_circuit')
n_juge = all_methodes.count('juge_llm')
n_fb = all_methodes.count('fallback_subterm')
n_none = all_methodes.count('no_candidates')
n_total = len(all_methodes)
print(f"\n   ⚡ Coupe-circuit : {n_cc}/{n_total} ({n_cc/n_total*100:.0f}%)")
print(f"   🧠 Juge LLM     : {n_juge}/{n_total} ({n_juge/n_total*100:.0f}%)")
print(f"   🔄 Fallback sub  : {n_fb}/{n_total} ({n_fb/n_total*100:.0f}%)")
print(f"   ❌ No candidates : {n_none}/{n_total} ({n_none/n_total*100:.0f}%)")

# Stats matching
n_exact = all_match_types.count('exact')
n_child = all_match_types.count('child')
n_parent = all_match_types.count('parent')
n_impl = all_match_types.count('implication')
print(f"\n   🔗 Matching hiérarchique :")
print(f"      EXACT  : {n_exact}")
print(f"      CHILD  : {n_child} ({SCORE_CHILD_MATCH:.0f}%)")
print(f"      PARENT : {n_parent} ({SCORE_PARENT_MATCH:.0f}%)")
print(f"      IMPL   : {n_impl} (auto-validé)")

🚀 Benchmark RAG Neurosymbolique : 75 évaluations
   5 participants × 15 cas


Pipeline RAG: 100%|██████████| 75/75 [12:18<00:00,  9.84s/it]


✅ Benchmark terminé en 738s (12.3 min)
   Évaluations réussies : 73 / 75
   Score moyen (validants)  : 81.4%
   Score médian             : 100.0%
   Écart-type               : 28.7%
   Latence moy/cas          : 10.1s

   ⚡ Coupe-circuit : 156/381 (41%)
   🧠 Juge LLM     : 199/381 (52%)
   🔄 Fallback sub  : 26/381 (7%)
   ❌ No candidates : 0/381 (0%)

   🔗 Matching hiérarchique :
      EXACT  : 83
      CHILD  : 17 (90%)
      PARENT : 17 (40%)
      IMPL   : 0 (auto-validé)


## 📊 Cellule 4 — Visualisation & Audit Clinique (Dark Theme)

### A) Heatmap Participant × Cas (fond sombre)
### B) Tableau d'audit — Validant X/X + Descripteur X/X + code couleur match_type
### C) Classement des participants + Difficulté par cas
### D) Détail des cas à 0%
### E) Métriques finales

In [10]:
# ============================================================
# CELLULE 4 — Visualisation & Audit Clinique
# ============================================================
from IPython.display import display, HTML

# ═══════════════════════════════════════════════════════════════
# Palette dark-friendly
# ═══════════════════════════════════════════════════════════════
COLORS = {
    'exact':  '#4CAF50',   # Vert vif
    'child':  '#FF9800',   # Orange
    'parent': '#F44336',   # Rouge
    'impl':   '#2196F3',   # Bleu
    'miss':   '#9E9E9E',   # Gris
    'bg_dark':    '#1e1e1e',
    'bg_row':     '#2d2d2d',
    'bg_row_alt': '#252525',
    'bg_header':  '#333333',
    'text':       '#e0e0e0',
    'text_dim':   '#999999',
}

# ═══════════════════════════════════════════════════════════════
# A) HEATMAP — Score final (Participant × Cas) — DARK THEME
# ═══════════════════════════════════════════════════════════════

def heatmap_bg(val):
    """Couleur de fond dark-friendly selon le score."""
    if pd.isna(val) or not isinstance(val, (int, float)):
        return f'background-color: {COLORS["bg_dark"]}; color: {COLORS["text_dim"]}'
    if val >= 90:  return 'background-color: #1b5e20; color: #a5d6a7; font-weight: bold'
    if val >= 70:  return 'background-color: #33691e; color: #c5e1a5'
    if val >= 50:  return 'background-color: #4e342e; color: #ffcc80'
    if val >= 20:  return 'background-color: #b71c1c; color: #ef9a9a'
    return 'background-color: #4a0000; color: #ff8a80; font-weight: bold'

pivot = df_valid.pivot_table(index='participant', columns='cas', values='score_final', aggfunc='first')
pivot['Moyenne'] = pivot.mean(axis=1).round(1)
mean_row = pivot.mean(axis=0).round(1)
mean_row.name = 'MOYENNE'
pivot = pd.concat([pivot, mean_row.to_frame().T])

print("═" * 90)
print("A) HEATMAP — Score final (%) par Participant × Cas  [Validants uniquement, sans bonus]")
print("═" * 90)
display(pivot.style
    .map(heatmap_bg)
    .set_caption("🎯 Scores finaux — Validants uniquement (%) — Participant × Cas")
    .set_properties(**{'text-align': 'center', 'font-size': '12px', 'border': '1px solid #444'})
    .format(precision=1, na_rep='--')
    .set_table_styles([
        {'selector': 'th', 'props': f'background-color: {COLORS["bg_header"]}; color: {COLORS["text"]}; border: 1px solid #444;'},
        {'selector': 'caption', 'props': f'color: {COLORS["text"]}; font-size: 14px; font-weight: bold;'},
    ])
)

# ═══════════════════════════════════════════════════════════════
# B) TABLEAU D'AUDIT — Validant X/X + Descripteur X/X + Match Types colorés
# ═══════════════════════════════════════════════════════════════

print(f"\n{'═'*90}")
print("B) TABLEAU D'AUDIT — Validants & Descripteurs avec code couleur")
print("═" * 90)

# Légende
display(HTML(f"""
<div style="background:{COLORS['bg_dark']}; padding:10px; border-radius:8px; margin-bottom:10px; font-family:monospace;">
  <b style="color:{COLORS['text']}">Légende match_type :</b>
  <span style="color:{COLORS['exact']}; font-weight:bold"> ● EXACT (100%)</span> &nbsp;
  <span style="color:{COLORS['child']}; font-weight:bold"> ● CHILD ({SCORE_CHILD_MATCH:.0f}%)</span> &nbsp;
  <span style="color:{COLORS['parent']}; font-weight:bold"> ● PARENT ({SCORE_PARENT_MATCH:.0f}%)</span> &nbsp;
  <span style="color:{COLORS['impl']}; font-weight:bold"> ● IMPLICATION (auto)</span> &nbsp;
  <span style="color:{COLORS['miss']}; font-weight:bold"> ● MANQUÉ (0%)</span>
</div>
"""))

# Construire le HTML du tableau d'audit
html_rows = []
for i, (_, row) in enumerate(df_valid.iterrows()):
    r = all_results[row.name]
    bg = COLORS['bg_row'] if i % 2 == 0 else COLORS['bg_row_alt']

    # Construire la colonne "Validants" avec code couleur
    val_parts = []
    for gn, role in zip(row['golden_names'], row['golden_roles']):
        if role != 'validant':
            continue
        mt = r['match_types'].get(gn, 'missing')
        color = COLORS.get(mt, COLORS['miss'])
        symbol = {'exact': '✅', 'child': '🟠', 'parent': '🔴', 'implication': '🔵'}.get(mt, '❌')
        val_parts.append(f'<span style="color:{color}" title="{mt.upper()}">{symbol} {gn}</span>')

    # Construire la colonne "Descripteurs" avec code couleur
    desc_parts = []
    for gn, role in zip(row['golden_names'], row['golden_roles']):
        if role != 'descripteur':
            continue
        mt = r['match_types'].get(gn, 'missing')
        color = COLORS.get(mt, COLORS['miss'])
        symbol = {'exact': '✅', 'child': '🟠', 'parent': '🔴', 'implication': '🔵'}.get(mt, '❌')
        desc_parts.append(f'<span style="color:{color}" title="{mt.upper()}">{symbol} {gn}</span>')

    vf, vt = r['validant_found'], r['validant_total']
    df_, dt = r['descripteur_found'], r['descripteur_total']
    score = r['score_final_pct']

    # Couleur du score
    if score >= 90: sc = COLORS['exact']
    elif score >= 50: sc = COLORS['child']
    else: sc = COLORS['parent']

    html_rows.append(f"""
    <tr style="background:{bg}">
      <td style="color:{COLORS['text']}; padding:4px 8px; white-space:nowrap">{row['participant']}</td>
      <td style="color:{COLORS['text']}; padding:4px 8px; text-align:center">{row['cas']}</td>
      <td style="color:{COLORS['text_dim']}; padding:4px 8px; font-size:10px; max-width:200px; overflow:hidden; text-overflow:ellipsis; white-space:nowrap">{str(row['texte_etudiant'])[:80]}</td>
      <td style="color:{COLORS['text']}; padding:4px 8px; font-weight:bold; text-align:center">{vf}/{vt}</td>
      <td style="padding:4px 8px; font-size:11px">{'<br>'.join(val_parts) if val_parts else '<span style="color:#666">—</span>'}</td>
      <td style="color:{COLORS['text']}; padding:4px 8px; font-weight:bold; text-align:center">{df_}/{dt}</td>
      <td style="padding:4px 8px; font-size:11px">{'<br>'.join(desc_parts) if desc_parts else '<span style="color:#666">—</span>'}</td>
      <td style="color:{sc}; padding:4px 8px; text-align:center; font-weight:bold; font-size:14px">{score:.0f}%</td>
    </tr>""")

audit_html = f"""
<table style="border-collapse:collapse; width:100%; font-family:monospace; font-size:12px; background:{COLORS['bg_dark']}; border:1px solid #444">
  <thead>
    <tr style="background:{COLORS['bg_header']}">
      <th style="color:{COLORS['text']}; padding:6px 8px; border:1px solid #444">Participant</th>
      <th style="color:{COLORS['text']}; padding:6px 8px; border:1px solid #444">Cas</th>
      <th style="color:{COLORS['text']}; padding:6px 8px; border:1px solid #444">Texte étudiant</th>
      <th style="color:{COLORS['text']}; padding:6px 8px; border:1px solid #444">Validant</th>
      <th style="color:{COLORS['text']}; padding:6px 8px; border:1px solid #444">Détail validants</th>
      <th style="color:{COLORS['text']}; padding:6px 8px; border:1px solid #444">Descripteur</th>
      <th style="color:{COLORS['text']}; padding:6px 8px; border:1px solid #444">Détail descripteurs</th>
      <th style="color:{COLORS['text']}; padding:6px 8px; border:1px solid #444">Score</th>
    </tr>
  </thead>
  <tbody>
    {''.join(html_rows)}
  </tbody>
</table>
"""
display(HTML(audit_html))

# ═══════════════════════════════════════════════════════════════
# C) CLASSEMENT + DIFFICULTÉ
# ═══════════════════════════════════════════════════════════════

print(f"\n{'═'*90}")
print("C) CLASSEMENT DES PARTICIPANTS")
print("═" * 90)

p_stats = df_valid.groupby('participant').agg(
    score_moyen=('score_final', 'mean'),
    score_median=('score_final', 'median'),
    n_parfaits=('score_final', lambda x: (x >= 100).sum()),
    n_zeros=('score_final', lambda x: (x == 0).sum()),
    latence_moy=('latence_s', 'mean'),
).round(1).sort_values('score_moyen', ascending=False)

display(p_stats.style
    .map(heatmap_bg, subset=['score_moyen', 'score_median'])
    .set_caption("🏆 Classement des participants")
    .set_properties(**{'text-align': 'center', 'border': '1px solid #444'})
    .set_table_styles([
        {'selector': 'th', 'props': f'background-color: {COLORS["bg_header"]}; color: {COLORS["text"]}; border: 1px solid #444;'},
        {'selector': 'td', 'props': f'background-color: {COLORS["bg_row"]}; color: {COLORS["text"]};'},
        {'selector': 'caption', 'props': f'color: {COLORS["text"]}; font-size: 14px; font-weight: bold;'},
    ])
)

print(f"\n{'═'*90}")
print("DIFFICULTÉ PAR CAS (du plus difficile au plus facile)")
print("═" * 90)

c_stats = df_valid.groupby(['cas', 'diagnostic_principal']).agg(
    score_moyen=('score_final', 'mean'),
    score_min=('score_final', 'min'),
    score_max=('score_final', 'max'),
    ecart_type=('score_final', 'std'),
    latence_moy=('latence_s', 'mean'),
).round(1).sort_values('score_moyen', ascending=True)

display(c_stats.style
    .map(heatmap_bg, subset=['score_moyen'])
    .set_caption("📈 Difficulté par cas ECG")
    .set_properties(**{'text-align': 'center', 'border': '1px solid #444'})
    .set_table_styles([
        {'selector': 'th', 'props': f'background-color: {COLORS["bg_header"]}; color: {COLORS["text"]}; border: 1px solid #444;'},
        {'selector': 'td', 'props': f'background-color: {COLORS["bg_row"]}; color: {COLORS["text"]};'},
        {'selector': 'caption', 'props': f'color: {COLORS["text"]}; font-size: 14px; font-weight: bold;'},
    ])
)

# ═══════════════════════════════════════════════════════════════
# D) DÉTAIL DES CAS À 0%
# ═══════════════════════════════════════════════════════════════

zeros = df_valid[df_valid['score_final'] == 0]
print(f"\n{'═'*90}")
print(f"D) DÉTAIL DES {len(zeros)} CAS À 0% (investigation)")
print("═" * 90)

for _, row in zeros.iterrows():
    print(f"\n[0%] {row['participant']} — Cas {row['cas']} — {row['diagnostic_principal']}")
    print(f"   TEXTE   : \"{row['texte_etudiant'][:120]}\"")
    print(f"   ATTENDU : {row['golden_names']}")
    print(f"   IA      : {row['concepts_ia'] or '(rien trouvé)'}")
    print(f"   MANQUANT: {row['missing']}")

# ═══════════════════════════════════════════════════════════════
# E) MÉTRIQUES FINALES
# ═══════════════════════════════════════════════════════════════

print(f"\n{'═'*90}")
print("📊 MÉTRIQUES FINALES — Pipeline RAG Neurosymbolique (Score = Validants uniquement)")
print("═" * 90)
print(f"   Score moyen global  : {df_valid['score_final'].mean():.1f}%")
print(f"   Score médian        : {df_valid['score_final'].median():.1f}%")
print(f"   Écart-type          : {df_valid['score_final'].std():.1f}%")
print(f"   Min / Max           : {df_valid['score_final'].min():.1f}% / {df_valid['score_final'].max():.1f}%")
print(f"   Cas parfaits (100%) : {(df_valid['score_final'] >= 100).sum()} / {len(df_valid)}")
print(f"   Cas à 0%            : {(df_valid['score_final'] == 0).sum()} / {len(df_valid)}")
print(f"   Latence totale      : {df_valid['latence_s'].sum():.0f}s ({df_valid['latence_s'].sum()/60:.1f} min)")
print(f"   Latence moy/cas     : {df_valid['latence_s'].mean():.1f}s")

# Distribution
bins = [0, 20, 40, 60, 80, 101]
labels = ['0-19%', '20-39%', '40-59%', '60-79%', '80-100%']
df_valid['tranche'] = pd.cut(df_valid['score_final'], bins=bins, labels=labels, right=False)
dist = df_valid['tranche'].value_counts().sort_index()
n = len(df_valid)
print(f"\n   Distribution des scores :")
for tranche, count in dist.items():
    bar = '█' * int(count / n * 40) + '░' * (40 - int(count / n * 40))
    print(f"      {tranche:>8s} : {bar} {count:2d} ({count/n*100:.0f}%)")

print(f"\n🎯 Verdict : Précision globale (validants) = {df_valid['score_final'].mean():.1f}%")

══════════════════════════════════════════════════════════════════════════════════════════
A) HEATMAP — Score final (%) par Participant × Cas  [Validants uniquement, sans bonus]
══════════════════════════════════════════════════════════════════════════════════════════


cas,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,Moyenne
ECG-1I3Q,100.0,100.0,40.0,32.0,45.0,95.0,100.0,100.0,0.0,100.0,100.0,100.0,100.0,90.0,100.0,80.1
ECG-3RMP,90.0,100.0,100.0,100.0,--,100.0,100.0,40.0,100.0,40.0,--,90.0,100.0,90.0,70.0,86.2
ECG-7512,90.0,0.0,100.0,100.0,95.0,50.0,80.0,100.0,0.0,40.0,100.0,100.0,100.0,100.0,70.0,75.0
ECG-DFLC,90.0,100.0,100.0,100.0,90.0,50.0,80.0,100.0,100.0,40.0,100.0,40.0,100.0,100.0,20.0,80.7
ECG-IDXQ,100.0,100.0,100.0,32.0,90.0,100.0,66.7,100.0,100.0,40.0,100.0,90.0,100.0,100.0,70.0,85.9
MOYENNE,94.0,80.0,88.0,72.8,80.0,79.0,85.3,88.0,60.0,52.0,100.0,84.0,100.0,96.0,66.0,81.6



══════════════════════════════════════════════════════════════════════════════════════════
B) TABLEAU D'AUDIT — Validants & Descripteurs avec code couleur
══════════════════════════════════════════════════════════════════════════════════════════


Participant,Cas,Texte étudiant,Validant,Détail validants,Descripteur,Détail descripteurs,Score
ECG-7512,1,Sinusal qrs fins P bifide,1/1,🟠 ECG normal,1/1,✅ Bloc interatrial,90%
ECG-7512,2,bav 1 hbag bbd,0/1,❌ BAV complet,0/1,❌ Echappement ventriculaire,0%
ECG-7512,3,fibrillation atriale,1/1,✅ Fibrillation atriale,0/1,❌ Repolarisation précoce,100%
ECG-7512,4,microvoltage,1/1,✅ Microvoltage,1/3,🟠 Amylose❌ BAV de type 1❌ Perte des onde Q septales,100%
ECG-7512,5,hyperkaliemie qrs fins onde amble bav complet,2/2,✅ Hyperkaliémie🟠 BAV de haut grade,0/0,—,95%
ECG-7512,6,stimulation atriale,1/2,✅ Stimulation atriale❌ Bloc fasciculaire antérieur gauche,0/1,❌ Bloc intraventriculaire aspécifique,50%
ECG-7512,7,bbd et hbag bav 1,3/3,🔴 Bloc de branche droit complet✅ BAV de type 1✅ Bloc fasciculaire antérieur gauche,0/0,—,80%
ECG-7512,8,flutter commun qrs normaux,1/1,✅ Flutter droit typique,0/0,—,100%
ECG-7512,9,bav 2 mobitz 1,0/1,❌ BAV 2 Mobitz 2,0/2,❌ Bloc de branche droit❌ Bloc fasciculaire postérieur gauche,0%
ECG-7512,10,bloc de branche gauche,1/1,🔴 Bloc de branche gauche complet,0/2,❌ Rythme sinusal❌ PR normal,40%



══════════════════════════════════════════════════════════════════════════════════════════
C) CLASSEMENT DES PARTICIPANTS
══════════════════════════════════════════════════════════════════════════════════════════


,score_moyen,score_median,n_parfaits,n_zeros,latence_moy
participant,,,,,
ECG-3RMP,86.200000,100.000000,7,0,11.000000
ECG-IDXQ,85.900000,100.000000,9,0,18.100000
ECG-DFLC,80.700000,100.000000,8,0,7.800000
ECG-1I3Q,80.100000,100.000000,9,1,10.700000
ECG-7512,75.000000,95.000000,7,2,3.100000



══════════════════════════════════════════════════════════════════════════════════════════
DIFFICULTÉ PAR CAS (du plus difficile au plus facile)
══════════════════════════════════════════════════════════════════════════════════════════


,,score_moyen,score_min,score_max,ecart_type,latence_moy
cas,diagnostic_principal,,,,,
10,Bloc de branche gauche complet,52.000000,40.000000,100.000000,26.800000,4.900000
9,BAV 2 Mobitz 2,60.000000,0.000000,100.000000,54.800000,11.200000
15,Bloc de branche gauche complet,66.000000,20.000000,100.000000,28.800000,6.600000
4,Microvoltage,72.800000,32.000000,100.000000,37.200000,13.000000
6,Stimulation atriale,79.000000,50.000000,100.000000,26.600000,12.500000
5,Hyperkaliémie,80.000000,45.000000,95.000000,23.500000,12.300000
2,BAV complet,80.000000,0.000000,100.000000,44.700000,7.900000
12,Syndrome coronarien à la phase aigue avec sus-décalage du segment ST,84.000000,40.000000,100.000000,25.100000,12.300000
7,Bloc de branche droit complet,85.300000,66.700000,100.000000,14.400000,10.400000



══════════════════════════════════════════════════════════════════════════════════════════
D) DÉTAIL DES 3 CAS À 0% (investigation)
══════════════════════════════════════════════════════════════════════════════════════════

[0%] ECG-7512 — Cas 2 — BAV complet
   TEXTE   : "bav 1 hbag bbd"
   ATTENDU : ['BAV complet', 'Echappement ventriculaire']
   IA      : BAV de type 1 | Bloc fasciculaire antérieur gauche | Bloc de branche droit
   MANQUANT: BAV complet | Echappement ventriculaire

[0%] ECG-7512 — Cas 9 — BAV 2 Mobitz 2
   TEXTE   : "bav 2 mobitz 1"
   ATTENDU : ['BAV 2 Mobitz 2', 'Bloc de branche droit', 'Bloc fasciculaire postérieur gauche']
   IA      : BAV de type 2 | BAV 2 Mobitz 1
   MANQUANT: BAV 2 Mobitz 2 | Bloc de branche droit | Bloc fasciculaire postérieur gauche

[0%] ECG-1I3Q — Cas 9 — BAV 2 Mobitz 2
   TEXTE   : "Rythme sinusal axe gauche hbag bbdt complet 
Bav1 
Bav2M1 et BAV2/1"
   ATTENDU : ['BAV 2 Mobitz 2', 'Bloc de branche droit', 'Bloc fasciculaire postérieur 